In [1]:
import numpy as np
import sys
import copy
import random

In [ ]:
def read_instance(filename):
    """
    Lê o arquivo de instância e retorna os seguintes dados:
      - n_voos: número de voos
      - m_runways: número de pistas (runways)
      - r: array de horários de liberação dos voos
      - c: array de valores complementares (se aplicável)
      - p: array de penalidades por minuto para cada voo
      - t: matriz de tempos obrigatórios de espera entre os voos
       
    O formato do arquivo deve ser:
      <n_voos>
      <m_runways>
      <array r>
      <array c>
      <array p>
      <linhas restantes: cada linha representa uma linha da matriz t>
    """
    with open(filename, 'r') as f:
        # Lê todas as linhas, removendo espaços extras e ignorando linhas vazias
        lines = [line.strip() for line in f if line.strip()]
    
    # Extraindo os parâmetros principais
    try:
        n_voos = int(lines.pop(0))
        m_runways = int(lines.pop(0))
    except ValueError as e:
        raise ValueError("Erro ao converter número de voos ou número de pistas para inteiro.") from e
    
    # Para evitar problemas com múltiplos espaços, use split() sem argumento.
    try:
        r = np.array([int(v) for v in lines.pop(0).split()])
        c = np.array([int(v) for v in lines.pop(0).split()])
        p = np.array([int(v) for v in lines.pop(0).split()])
    except ValueError as e:
        raise ValueError("Erro ao converter os arrays 'r', 'c' ou 'p' para inteiros.") from e
    
    # O que restou são as linhas da matriz t
    t = []
    for l in lines:
        # Usa split() sem argumento para separar por espaços em branco (qualquer quantidade)
        row = [int(v) for v in l.split() if v]
        t.append(row)
    
    # Converte a lista de listas em um array NumPy para facilitar operações posteriores
    t = np.array(t)
    
    return n_voos, m_runways, r, c, p, t

In [3]:
filename = '/home/gabrielluccav/projects/APA/instances/n3m10A.txt'

In [4]:
n_voos, m_runways, r, c, p, t = read_instance(filename)

In [5]:
def compute_runway_schedule(runway, r, t):
    """
    Dada a lista de voos (índices) atribuídos a uma pista,
    calcula os horários de início e o custo (multa) para essa pista.
    A regra aqui é: o primeiro voo é programado exatamente no horário r[i].
    Para o voo j (após o voo i), o horário de início é:
      start_time(j) = max(r[j], start_time(i) + t[i][j])
    onde t[i][j] é o tempo obrigatório de espera (se voo j vier imediatamente após i).
    Retorna:
      start_times: uma lista com o horário de início para cada voo nessa pista.
      custo: soma da multa para todos os voos da pista, onde multa de voo i = p[i] * (start_time[i] - r[i]).
    Nota: Os parâmetros r (lista de liberação) e t (matriz) são considerados com índices correspondentes aos voos (0-indexed).
    """
    start_times = []
    # Para o primeiro voo
    first = runway[0]
    current_start = r[first]
    start_times.append(current_start)
    
    # Para os demais voos na mesma pista
    for idx in range(1, len(runway)):
        prev = runway[idx - 1]
        curr = runway[idx]
        # O voo atual só pode começar após:
        # (a) Seu horário de liberação, ou (b) a conclusão (start_time + waiting_time) do voo anterior.
        current_start = max(r[curr], start_times[-1] + t[prev][curr])
        start_times.append(current_start)
    
    return start_times

In [6]:
def compute_total_cost(solution, r, p, t):
    """
    Dada uma solução (lista de listas de voos, cada lista representa a sequência de voos em uma pista),
    calcula o custo total, isto é, a soma das multas de atraso de cada voo.
    Para cada pista, para cada voo i:
      - O horário de início é calculado com compute_runway_schedule.
      - A multa é: p[i] * (start_time[i] - r[i])
    Retorna:
      total_cost: valor numérico da soma das multas.
    """
    total_cost = 0
    for runway in solution:
        if len(runway) == 0:
            continue
        start_times = compute_runway_schedule(runway, r, t)
        for flight, start in zip(runway, start_times):
            delay = start - r[flight]
            total_cost += p[flight] * delay
    return total_cost

In [7]:
def initial_solution(n_voos, m_runways, r):
    """
    Gera uma solução inicial simples.
    Aqui, ordenamos os voos pela hora de liberação (r) e distribuímos os voos em forma “round-robin”
    entre as pistas.
    Retorna:
      solution: lista com m_runways listas, cada uma com os índices (0-indexados) dos voos atribuídos.
    """
    flights = list(range(n_voos))
    # Ordenar voos pela liberação (opcional – pode-se usar outras heurísticas construtivas)
    flights.sort(key=lambda i: r[i])
    solution = [[] for _ in range(m_runways)]
    for idx, flight in enumerate(flights):
        runway = idx % m_runways
        solution[runway].append(flight)
    return solution

In [16]:
r

array([56, 15, 34, 61, 29, 50, 65, 12, 61, 31])

In [24]:
flights = list(range(n_voos))
# Ordenar voos pela liberação (opcional – pode-se usar outras heurísticas construtivas)
flights.sort(key=lambda i: r[i])
for idx, flight in enumerate(flights):
    print(idx)
    print(idx % m_runways)
    print()

0
0

1
1

2
2

3
0

4
1

5
2

6
0

7
1

8
2

9
0



In [9]:
print({'n_voos': n_voos,
       'm_runways': m_runways,
       'r': r
       })

{'n_voos': 10, 'm_runways': 3, 'r': array([56, 15, 34, 61, 29, 50, 65, 12, 61, 31])}


In [11]:
sol = initial_solution(n_voos, m_runways, r)
sol

[[7, 9, 0, 6], [1, 2, 3], [4, 5, 8]]

In [9]:
def neighbor_swap_in_runway(solution):
    """
    Gera uma nova solução realizando uma troca de dois voos na mesma pista.
    Seleciona aleatoriamente uma pista com ao menos dois voos e troca dois voos nessa pista.
    Retorna:
      nova_sol: solução modificada.
    """
    new_solution = copy.deepcopy(solution)
    # Escolher aleatoriamente uma pista com ao menos 2 voos
    candidate_indices = [i for i, runway in enumerate(new_solution) if len(runway) >= 2]
    if not candidate_indices:
        return new_solution
    runway_index = random.choice(candidate_indices)
    runway = new_solution[runway_index]
    i, j = random.sample(range(len(runway)), 2)
    runway[i], runway[j] = runway[j], runway[i]
    return new_solution

In [10]:
n_sol = neighbor_swap_in_runway(sol)
n_sol

[[7, 9, 0, 6], [1, 2, 3], [4, 8, 5]]

In [11]:
def neighbor_move_between_runways(solution):
    """
    Gera uma nova solução realizando o movimento de um voo entre pistas:
    Remove aleatoriamente um voo de uma pista e insere-o em uma posição aleatória de outra pista.
    Retorna:
      nova_sol: solução modificada.
    """
    new_solution = copy.deepcopy(solution)
    # Seleciona uma pista de onde retirar (que não seja vazia)
    source_indices = [i for i, runway in enumerate(new_solution) if len(runway) > 0]
    if len(source_indices) < 1:
        return new_solution
    source = random.choice(source_indices)
    flight = new_solution[source].pop(random.randrange(len(new_solution[source])))
    # Seleciona uma pista diferente para inserir
    target_indices = [i for i in range(len(new_solution)) if i != source]
    if not target_indices:  # caso só exista uma pista
        new_solution[source].append(flight)
        return new_solution
    target = random.choice(target_indices)
    insertion_index = random.randrange(len(new_solution[target]) + 1)
    new_solution[target].insert(insertion_index, flight)
    return new_solution

In [12]:
neighbor_move_between_runways(sol)

[[7, 9, 0, 5, 6], [1, 2, 3], [4, 8]]

In [ ]:
def neighbor_reinsert_in_runway(solution):
    """
    Gera uma nova solução realizando a reinserção (ou reordenação) de um voo dentro da mesma pista.
    Esse movimento consiste em remover um voo de sua posição atual em uma pista e reinseri-lo em outra posição,
    alterando assim a ordem dos voos daquela pista.
    
    Retorna:
      new_solution: solução modificada com a operação aplicada.
    """
    new_solution = copy.deepcopy(solution)
    # Seleciona os índices das pistas com pelo menos 2 voos (para que a reinserção tenha efeito)
    candidate_indices = [i for i, runway in enumerate(new_solution) if len(runway) >= 2]
    if not candidate_indices:
        return new_solution  # Se nenhuma pista tiver pelo menos 2 voos, não há movimento a fazer

    # Seleciona uma pista aleatoriamente dentre as candidatas
    runway_index = random.choice(candidate_indices)
    runway = new_solution[runway_index]

    # Seleciona aleatoriamente um voo para ser removido (índice original)
    original_index = random.randrange(len(runway))
    flight = runway.pop(original_index)

    # Escolhe uma nova posição para reinserir o voo; se possível, que seja diferente da posição original
    if len(runway) == 0:
        new_position = 0
    else:
        new_position = random.randrange(len(runway) + 1)
        while len(runway) > 1 and new_position == original_index:
            new_position = random.randrange(len(runway) + 1)
    
    runway.insert(new_position, flight)
    return new_solution

In [20]:
neighbor_reinsert_in_runway(sol)

[[7, 9, 0, 6], [2, 3, 1], [4, 5, 8]]

In [21]:
def vnd(initial_solution, r, p, t):
    """
    Implementa o algoritmo Variable Neighborhood Descent (VND).
    Usa uma lista de funções de movimentação de vizinhança (no exemplo, três movimentos).
    A cada iteração, tenta melhorar a solução atual aplicando a vizinhança corrente.
    Se encontrar uma melhoria, reinicia a sequência de vizinhanças; caso contrário, passa para a próxima.
    
    Parâmetros:
      - initial_solution: solução inicial (lista de listas, cada uma representa os voos em uma pista)
      - r, p, t: dados da instância (array r, array p e matriz t)
    
    Retorna uma tupla (best_solution, best_cost).
    """
    neighborhood_functions = [
        neighbor_swap_in_runway,
        neighbor_move_between_runways,
        neighbor_reinsert_in_runway
    ]
    
    current_solution = copy.deepcopy(initial_solution)
    best_cost = compute_total_cost(current_solution, r, p, t)
    k = 0
    while k < len(neighborhood_functions):
        candidate_solution = neighborhood_functions[k](current_solution)
        candidate_cost = compute_total_cost(candidate_solution, r, p, t)
        # Se a solução melhorou, reinicia a lista de vizinhanças
        if candidate_cost < best_cost:
            current_solution = candidate_solution
            best_cost = candidate_cost
            k = 0  # reinicia para a primeira vizinhança
        else:
            k += 1  # passa para a próxima vizinhança
    return current_solution, best_cost

In [28]:
vnd(sol, r, p, t)

([[7, 6, 0, 9], [1, 2, 3], [4, 5, 8]], np.int64(4905))

In [ ]:
def local_search(solution, r, p, t, max_iter=1000):
    """
    Busca local iterativa que tenta melhorar a solução utilizando movimentos de vizinhança.
    Tenta repetidamente movimentos (swap dentro da mesma pista ou mover entre pistas) e aceita o movimento
    se houver melhora (redução no custo total).
    Parâmetro max_iter define o número máximo de iterações sem melhora para finalizar.
    Retorna:
      best_solution: a melhor solução encontrada.
      best_cost: custo total da melhor solução.
    """
    best_solution = copy.deepcopy(solution)
    best_cost = compute_total_cost(best_solution, r, p, t)
    
    iter_without_improve = 0
    while iter_without_improve < max_iter:
        # Escolher aleatoriamente qual movimento usar
        rnd = random.random()
        if rnd < 0.33:
            new_solution = neighbor_swap_in_runway(best_solution)
        elif rnd > 0.66:
            new_solution = neighbor_reinsert_in_runway(best_solution)
        else:
            new_solution = neighbor_move_between_runways(best_solution)
        new_cost = compute_total_cost(new_solution, r, p, t)
        if new_cost < best_cost:
            best_cost = new_cost
            best_solution = new_solution
            iter_without_improve = 0
        else:
            iter_without_improve += 1
    return best_solution, best_cost

In [18]:
local_search(sol, r, p, t)

([[2, 9, 6], [1, 0, 3], [7, 4, 5, 8]], np.int64(704))

In [ ]:
def write_solution(filename, solution, total_cost):
    """
    Escreve o arquivo de saída conforme o formato:
      Linha 1: total_cost
      Linha 2: lista de voos alocados na pista 1 (indices +1 para apresentar como 1-index)
      Linha 3: lista de voos alocados na pista 2
      etc.
    """
    with open(filename, 'w') as f:
        f.write(f"1 {total_cost}\n")
        for i, runway in enumerate(solution):
            # Converter índices 0-indexados para 1-indexados
            flights_line = " ".join(str(flight + 1) for flight in runway)
            f.write(f"{i+2} {flights_line}\n")

In [ ]:
def main():
    if len(sys.argv) < 3:
        print("Uso: python projeto_final.py <arquivo_entrada> <arquivo_saida>")
        sys.exit(1)
    
    input_file = sys.argv[1]
    output_file = sys.argv[2]
    
    # Ler a instância
    n_voos, m_runways, r, c, p, t = read_instance(input_file)
    print(f"Número de voos: {n_voos}, Número de pistas: {m_runways}")
    
    # Gerar solução inicial
    sol = initial_solution(n_voos, m_runways, r)
    initial_cost = compute_total_cost(sol, r, p, t)
    print(f"Solução inicial (custo = {initial_cost}): {sol}")
    
    # Melhorar usando busca local
    best_sol, best_cost = local_search(sol, r, p, t, max_iter=1000)
    print(f"Melhor solução encontrada (custo = {best_cost}): {best_sol}")
    
    # Escrever solução em arquivo de saída
    write_solution(output_file, best_sol, best_cost)
    print(f"Solução escrita em {output_file}")

if __name__ == '__main__':
    main()

In [6]:
n_voos, m_runways, r, c, p, t = read_instance('/home/gabrielluccav/projects/APA/instances/n3m10A.txt')
print(f"Número de voos: {n_voos}, Número de pistas: {m_runways}")

IndexError: list index out of range

In [ ]:
# Gerar solução inicial
sol = initial_solution(n_voos, m_runways, r)
initial_cost = compute_total_cost(sol, r, p, t)
print(f"Solução inicial (custo = {initial_cost}): {sol}")

# Melhorar usando busca local
best_sol, best_cost = local_search(sol, r, p, t, max_iter=1000)
print(f"Melhor solução encontrada (custo = {best_cost}): {best_sol}")

# Escrever solução em arquivo de saída
write_solution(output_file, best_sol, best_cost)
print(f"Solução escrita em {output_file}")